# Y-randomization validation

This notebook tests whether model performance is dependent on the real activity labels by repeatedly permuting pIC50 values and refitting the model. Randomized models should perform poorly relative to the non-randomized model.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

ROOT = Path("..").resolve()
DATA = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

X = pd.read_csv(DATA / "X_combined.csv")
y = pd.read_csv(DATA / "y_pIC50.csv")["pIC50"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X.shape, y.shape


In [ ]:
baseline_model = RandomForestRegressor(n_estimators=120, random_state=42, n_jobs=-1)
baseline_model.fit(X_train, y_train)
baseline_r2 = r2_score(y_test, baseline_model.predict(X_test))
baseline_r2


In [ ]:
n_permutations = 30
scores = []

for seed in range(n_permutations):
    y_permuted = y_train.sample(frac=1, random_state=seed).reset_index(drop=True)
    model = RandomForestRegressor(n_estimators=80, random_state=seed, n_jobs=-1)
    model.fit(X_train.reset_index(drop=True), y_permuted)
    scores.append(r2_score(y_test, model.predict(X_test)))

randomization_scores = pd.DataFrame({"Randomized_R2": scores})
randomization_scores.to_csv(RESULTS / "y_randomization_scores.csv", index=False)
randomization_scores.describe()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(randomization_scores["Randomized_R2"], bins=12, color="#64748B", edgecolor="white")
ax.axvline(baseline_r2, color="#BE123C", linewidth=2, label=f"Baseline R2 = {baseline_r2:.2f}")
ax.set_xlabel("R2 after pIC50 permutation")
ax.set_ylabel("Frequency")
ax.set_title("Y-randomization validation")
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS / "y_randomization_validation.png", dpi=300, bbox_inches="tight")
plt.show()
